# `nb02d`: Exploratory data analysis

We look at what the 344 penguin records show: one variable at a time, then in pairs, then all together. Every plot below is a view of their empirical distribution.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('data/penguins.csv')

In [ ]:
df.head()

In [ ]:
df.describe(include='all')

`describe` reports how many values are present in each column, which is where the missing entries show up.

In [ ]:
df[df.isna().any(axis=1)]

# Univariate analysis

The raw values, then the histogram, which is a marginal of the empirical distribution, then summary statistics.

In [ ]:
df['body_mass_g'].values

In [ ]:
df['body_mass_g'].plot(kind='hist', bins=30)
# plt.savefig('body_mass_histogram.png')

In [ ]:
df['body_mass_g'].describe()

In [ ]:
df['flipper_length_mm'].values

In [ ]:
df['flipper_length_mm'].plot(kind='hist', bins=30)
# plt.savefig('flipper_length_histogram.png')

In [ ]:
df['flipper_length_mm'].describe()

In [ ]:
df['species'].value_counts().plot(kind='bar')
# plt.savefig('species_counts.png')

# Bivariate analysis

Two variables at a time: a joint for two numerical variables, a conditional when one is categorical, a contingency table when both are.

In [ ]:
df[['body_mass_g', 'flipper_length_mm']].plot(kind='scatter', x='body_mass_g', y='flipper_length_mm')
# plt.savefig('body_mass_vs_flipper_length.png')

In [ ]:
df.hist(column='body_mass_g', by='species', bins=30)
# plt.savefig('body_mass_by_species.png')


In [ ]:
pd.crosstab(df['species'], df['island'])

Gentoo live only on Biscoe and Chinstrap only on Dream: species and island are dependent.

In [ ]:
df.corr(numeric_only=True)

In [ ]:
df.corr(numeric_only=True, method='spearman')

In [ ]:
from sklearn.metrics import mutual_info_score
mutual_info_score(df['species'], df['island'])

In [ ]:
import seaborn as sns
sns.pairplot(df, kind='scatter', corner=True)
# plt.savefig('pairplot.png')

# Multivariate analysis

All variables at once: colour by species, then project the four measurements down to two dimensions.

In [ ]:
sns.pairplot(df, kind='scatter', hue='species', corner=True)
# plt.savefig('pairplot_by_species.png')

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

data = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'species']].dropna().values
species = data[:, -1]
species = pd.Series(species).astype('category').cat.codes
data = data[:, :-1]

scaler = StandardScaler()
data = scaler.fit_transform(data)

pca = PCA(n_components=2)
data_2d = pca.fit_transform(data)   

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(1, 1, 1)

ax.scatter(data_2d[:, 0], data_2d[:, 1], c=species, cmap='viridis', alpha=0.7)
ax.set_xlabel('PCA1')
ax.set_ylabel('PCA2')

# plt.savefig('pca_penguins.png')


In [ ]:
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42)
data_tsne = tsne.fit_transform(data)

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(1, 1, 1)

ax.scatter(data_tsne[:, 0], data_tsne[:, 1], c=species, cmap='viridis', alpha=0.7)
ax.set_xlabel('t-SNE1')
ax.set_ylabel('t-SNE2')

# plt.savefig('tsne_penguins.png')

What this suggests, none of it established here:

- body mass is not a single population, since the sample mixes three species,
- body mass increases with flipper length,
- species and island are dependent, and measurements differ from group to group,
- the four measurements are strongly correlated, so fewer dimensions may be enough.

These are the models built from Lecture 4 onwards.